In [0]:
import importlib
import pandas as pd
import src.utils.fetch_lighthouse_data as fld
importlib.reload(fld)
from scipy.stats import percentileofscore


from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

API_KEY = dbutils.secrets.get(
    scope="site_speed_project",
    key="google_psi_api_key"
)

In [0]:
result = fld.fetch_data(api_key=API_KEY, url="https://www.hafuboti.com/")

In [0]:
df = spark.sql('''
SELECT * 
FROM workspace.`site-speed-recommendation`.combined
WHERE performance_score IS NOT NULL
''').toPandas()

In [0]:
print(df['performance_score'].describe())
print(result['performance_score'])
percentileofscore(df['performance_score'].values, result['performance_score'])

In [0]:
print_df = {}
for col in result.keys():
    print_df[col] = round(percentileofscore(df[col].values, result[col]),2)

pd.DataFrame(print_df, index=[0]).columns

In [0]:
feature_groups = {
    "page_weight_burden": [
        "total-byte-weight",
        "resource_total_bytes",
        "resource_total_requests"
    ],

    "js_size_burden": [
        "resource_script_bytes",
        "resource_script_requests",
        "unused-javascript_savings_bytes"
    ],

    "js_execution_burden": [
        "mainthread_scriptEvaluation",
        "mainthread_scriptParseCompile",
        "unused-javascript_savings_ms"
    ],

    "media_burden": [
        "resource_image_bytes",
        "resource_image_requests",
        "resource_media_bytes",
        "resource_media_requests"
    ],

    "third_party_burden": [
        "resource_third-party_bytes",
        "resource_third-party_requests"
    ],

    "dom_structure_burden": [
        "dom-size-insight",
        "mainthread_parseHTML"
    ],

    "rendering_burden": [
        "mainthread_styleLayout",
        "mainthread_paintCompositeRender"
    ],

    "network_server_burden": [
        "network-server-latency",
        "EXPERIMENTAL_TIME_TO_FIRST_BYTE"
    ],

    "css_burden": [
        "resource_stylesheet_bytes",
        "resource_stylesheet_requests",
        "unused-css-rules_savings_bytes",
        "unused-css-rules_savings_ms"
    ]
}
recommendations = {
    "js_size_burden": [
        "Reduce unused JavaScript",
        "Split large JavaScript bundles",
        "Defer non-critical scripts"
    ],
    "js_execution_burden": [
        "Reduce main-thread JavaScript execution",
        "Audit expensive scripts",
        "Delay non-critical third-party code"
    ],
    "media_burden": [
        "Compress large images",
        "Use next-gen image formats",
        "Lazy-load below-the-fold media"
    ],
    "third_party_burden": [
        "Audit third-party tags",
        "Remove unnecessary tracking scripts",
        "Load third-party scripts asynchronously"
    ],
    "network_server_burden": [
        "Improve server response time",
        "Use caching or a CDN",
        "Reduce backend latency"
    ],
    "dom_structure_burden": [
        "Reduce DOM size",
        "Simplify nested layout elements",
        "Avoid excessive client-side rendering"
    ],
    "rendering_burden": [
        "Reduce layout and paint work",
        "Simplify expensive visual effects",
        "Minimize layout shifts"
    ],
    "css_burden": [
        "Remove unused CSS",
        "Minify CSS",
        "Inline critical CSS where appropriate"
    ],
    "page_weight_burden": [
        "Reduce total page weight",
        "Remove unnecessary resources",
        "Prioritize critical assets"
    ]
}

In [0]:
new_df = df.dropna().copy()

In [0]:
burden_df = pd.DataFrame(index=new_df.index)

for score_name, cols in feature_groups.items():
    scaled_values = StandardScaler().fit_transform(new_df[cols])
    burden_df[score_name] = scaled_values.mean(axis=1)
    
X_cluster = StandardScaler().fit_transform(burden_df)

kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)

new_df["performance_cluster"] = kmeans.fit_predict(X_cluster)

cluster_profile = (
    burden_df
    .assign(cluster=new_df["performance_cluster"])
    .groupby("cluster")
    .mean()
)

cluster_profile

In [0]:
for cluster in cluster_profile.index:
    print(f"\nCluster {cluster}")
    print(cluster_profile.loc[cluster].sort_values(ascending=False).head(5))

In [0]:
X = burden_df[[
    "page_weight_burden",
    "js_size_burden",
    "js_execution_burden",
    "media_burden",
    "third_party_burden",
    "dom_structure_burden",
    "rendering_burden",
    "network_server_burden",
    "css_burden"
]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

results = []

for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)

    results.append({
        "k": k,
        "inertia": kmeans.inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
        "davies_bouldin": davies_bouldin_score(X_scaled, labels),
        "calinski_harabasz": calinski_harabasz_score(X_scaled, labels),
        "smallest_cluster_size": pd.Series(labels).value_counts().min(),
        "largest_cluster_size": pd.Series(labels).value_counts().max()
    })

cluster_results = pd.DataFrame(results)
cluster_results

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(cluster_results["k"], cluster_results["inertia"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

In [0]:
plt.figure(figsize=(8, 5))
plt.plot(cluster_results["k"], cluster_results["silhouette"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score by k")
plt.show()

In [0]:
plt.figure(figsize=(8, 5))
plt.plot(cluster_results["k"], cluster_results["davies_bouldin"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Davies-Bouldin Score")
plt.title("Davies-Bouldin Score by k")
plt.show()

In [0]:
for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)

    print(f"\nK = {k}")
    print(pd.Series(labels).value_counts().sort_index())

In [0]:
for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)

    profile = (
        X.assign(cluster=labels)
        .groupby("cluster")
        .mean()
    )

    print(f"\n========== K = {k} ==========")

    for cluster in profile.index:
        print(f"\nCluster {cluster}")
        print(profile.loc[cluster].sort_values(ascending=False).head(4))

In [0]:
df_clipped = new_df.copy()
numeric_cols = df_clipped.select_dtypes(include="number").columns

for col in numeric_cols:
    lower = df_clipped[col].quantile(0.00)
    upper = df_clipped[col].quantile(0.99)
    df_clipped[col] = df_clipped[col].clip(lower, upper)

In [0]:
burden_df = pd.DataFrame(index=df_clipped.index)

for score_name, cols in feature_groups.items():
    scaled_values = StandardScaler().fit_transform(df_clipped[cols])
    burden_df[score_name] = scaled_values.mean(axis=1)
    
X_cluster = StandardScaler().fit_transform(burden_df)

kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)

df_clipped["performance_cluster"] = kmeans.fit_predict(X_cluster)

cluster_profile = (
    burden_df
    .assign(cluster=df_clipped["performance_cluster"])
    .groupby("cluster")
    .mean()
)

cluster_profile

In [0]:
X = burden_df[[
    "page_weight_burden",
    "js_size_burden",
    "js_execution_burden",
    "media_burden",
    "third_party_burden",
    "dom_structure_burden",
    "rendering_burden",
    "network_server_burden",
    "css_burden"
]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

results = []

for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)

    results.append({
        "k": k,
        "inertia": kmeans.inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
        "davies_bouldin": davies_bouldin_score(X_scaled, labels),
        "calinski_harabasz": calinski_harabasz_score(X_scaled, labels),
        "smallest_cluster_size": pd.Series(labels).value_counts().min(),
        "largest_cluster_size": pd.Series(labels).value_counts().max()
    })

cluster_results = pd.DataFrame(results)
cluster_results

In [0]:
for k in [3, 4, 5]:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)

    profile = (
        X.assign(cluster=labels)
        .groupby("cluster")
        .mean()
    )

    sizes = pd.Series(labels).value_counts().sort_index()

    print(f"\n================ K = {k} ================")
    print("\nCluster sizes:")
    print(sizes)

    print("\nTop cluster signals:")
    for cluster in profile.index:
        print(f"\nCluster {cluster}")
        print(profile.loc[cluster].sort_values(ascending=False).head(5))

In [0]:
problem_sites = new_df[
    (df["largest-contentful-paint"] > 2500) |
    (df["cumulative-layout-shift"] > 0.1) |
    (df["INTERACTION_TO_NEXT_PAINT"] > 200)
].copy()
# new_df["fails_cwv"] = (
#     (df["largest-contentful-paint"] > 2500) |
#     (df["cumulative-layout-shift"] > 0.1) |
#     (
#         df["INTERACTION_TO_NEXT_PAINT"].notna() &
#         (df["INTERACTION_TO_NEXT_PAINT"] > 200)
#     ) |
#     (
#         df["INTERACTION_TO_NEXT_PAINT"].isna() &
#         (df["total-blocking-time"] > 200)
#     )
# )

numeric_cols = problem_sites.select_dtypes(include="number").columns
for col in numeric_cols:
    lower = problem_sites[col].quantile(0.00)
    upper = problem_sites[col].quantile(0.99)
    problem_sites[col] = problem_sites[col].clip(lower, upper)

burden_df = pd.DataFrame(index=problem_sites.index)

for score_name, cols in feature_groups.items():
    scaled_values = StandardScaler().fit_transform(problem_sites[cols])
    burden_df[score_name] = scaled_values.mean(axis=1)
    
X_cluster = StandardScaler().fit_transform(burden_df)

kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)

problem_sites["performance_cluster"] = kmeans.fit_predict(X_cluster)

cluster_profile = (
    burden_df
    .assign(cluster=problem_sites["performance_cluster"])
    .groupby("cluster")
    .mean()
)

cluster_profile

In [0]:
X = burden_df[[
    "page_weight_burden",
    "js_size_burden",
    "js_execution_burden",
    "media_burden",
    "third_party_burden",
    "dom_structure_burden",
    "rendering_burden",
    "network_server_burden",
    "css_burden"
]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

results = []

for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)

    results.append({
        "k": k,
        "inertia": kmeans.inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
        "davies_bouldin": davies_bouldin_score(X_scaled, labels),
        "calinski_harabasz": calinski_harabasz_score(X_scaled, labels),
        "smallest_cluster_size": pd.Series(labels).value_counts().min(),
        "largest_cluster_size": pd.Series(labels).value_counts().max()
    })

cluster_results = pd.DataFrame(results)
cluster_results